# Shipwrecks Data Science

Download the public shipwrecks dataset from Kaggle, print its categories (column names), and display the first five entries.

In [ ]:
%pip install -q kagglehub

import kagglehub
import pandas as pd

csv_path = kagglehub.dataset_download(
    "diaaessam/shipwrecks-sunk-ships",
    path="Shipwrecks or Sunk ships.csv",
)
shipwrecks = pd.read_csv(csv_path)

In [ ]:
print("Categories:")
for category in shipwrecks.columns:
    print(f"- {category}")

In [ ]:
print("First 5 entries:")
shipwrecks.head()

## Sort the shipwrecks by date

The original date text is preserved in `Sunk date`. Dates that pandas cannot interpret are stored as `NaT` and sorted to the bottom. Partial dates, such as `1941`, are interpreted as the beginning of that period.

In [ ]:
shipwrecks["Parsed sunk date"] = pd.to_datetime(
    shipwrecks["Sunk date"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)

shipwrecks_by_date = shipwrecks.sort_values(
    "Parsed sunk date",
    na_position="last",
).reset_index(drop=True)

shipwrecks_by_date[["Ship", "Sunk date", "Parsed sunk date"]].head(10)

## WWII sinkings before and after the naval Enigma breakthrough

There was no single moment when every Enigma system was broken. For this comparison, **9 May 1941** is used as the dividing line because HMS *Bulldog* captured U-110 and naval Enigma material on that date. The WWII range used here is 1 September 1939 through 2 September 1945. A partial date such as `1941` is parsed as 1 January 1941, so records without a complete date should be interpreted cautiously.

In [ ]:
wwii_start = pd.Timestamp("1939-09-01")
enigma_breakthrough = pd.Timestamp("1941-05-09")
wwii_end = pd.Timestamp("1945-09-02")

wwii_shipwrecks = shipwrecks_by_date[
    shipwrecks_by_date["Parsed sunk date"].between(wwii_start, wwii_end)
].copy()

before_enigma_breakthrough = wwii_shipwrecks[
    wwii_shipwrecks["Parsed sunk date"] < enigma_breakthrough
].copy()

after_enigma_breakthrough = wwii_shipwrecks[
    wwii_shipwrecks["Parsed sunk date"] >= enigma_breakthrough
].copy()

print(f"Before breakthrough: {len(before_enigma_breakthrough)} shipwrecks")
print(f"On/after breakthrough: {len(after_enigma_breakthrough)} shipwrecks")

In [ ]:
print("WWII sinkings before 9 May 1941:")
before_enigma_breakthrough[[
    "Ship", "Flag", "Sunk date", "Parsed sunk date", "Notes"
]]

In [ ]:
print("WWII sinkings on or after 9 May 1941:")
after_enigma_breakthrough[[
    "Ship", "Flag", "Sunk date", "Parsed sunk date", "Notes"
]]

## A second dataset: German U-boats

This public Kaggle dataset adds one row per U-boat, including patrol, sinking, tonnage, commander, wolfpack, and fate information. It gives us more useful measures of German operational performance than the shipwreck list alone. However, because many fields are totals per U-boat rather than individual dated attack attempts, we will first inspect its columns before deciding how strongly it can answer the Enigma question.

In [ ]:
uboats_csv_path = kagglehub.dataset_download(
    "cormac42/ww2-u-boats",
    path="uboats.csv",
)
uboats = pd.read_csv(uboats_csv_path)

print(f"Rows: {len(uboats):,}")
print(f"Columns: {len(uboats.columns)}")
print("\nColumn names:")
for column in uboats.columns:
    print(f"- {column}")

uboats.head()

## World map of shipwrecks

The map parses the degree-minute-second coordinates in the original dataset. It colors the points by broad date periods so WWII wrecks remain visually distinct. Hover over a point to see the ship, flag, original sinking date, coordinates, and parsed year.

In [ ]:
import re
import plotly.express as px

def dms_to_decimal(coordinate_text):
    """Convert a latitude/longitude DMS string into decimal coordinates."""
    if pd.isna(coordinate_text):
        return pd.Series({"Latitude": None, "Longitude": None})

    coordinate_parts = re.findall(
        r"([^NSEW]*)([NSEW])", str(coordinate_text).upper()
    )
    parsed_coordinates = {}

    for number_text, hemisphere in coordinate_parts:
        numbers = [float(value) for value in re.findall(r"\d+(?:\.\d+)?", number_text)]
        if not numbers:
            continue

        decimal_value = numbers[0]
        if len(numbers) > 1:
            decimal_value += numbers[1] / 60
        if len(numbers) > 2:
            decimal_value += numbers[2] / 3600
        if hemisphere in {"S", "W"}:
            decimal_value *= -1

        parsed_coordinates[hemisphere] = decimal_value

    latitude = parsed_coordinates.get("N", parsed_coordinates.get("S"))
    longitude = parsed_coordinates.get("E", parsed_coordinates.get("W"))
    return pd.Series({"Latitude": latitude, "Longitude": longitude})

coordinate_columns = shipwrecks["Coordinates"].apply(dms_to_decimal)
shipwrecks_mapped = pd.concat([shipwrecks.copy(), coordinate_columns], axis=1)
shipwrecks_mapped["Sinking year"] = shipwrecks_mapped["Parsed sunk date"].dt.year

shipwrecks_mapped["Date period"] = pd.cut(
    shipwrecks_mapped["Sinking year"],
    bins=[-float("inf"), 1899, 1938, 1945, 1999, float("inf")],
    labels=["Before 1900", "1900–1938", "WWII (1939–1945)", "1946–1999", "2000 or later"],
).astype("object").fillna("Unknown date")

map_data = shipwrecks_mapped.dropna(subset=["Latitude", "Longitude"]).copy()
print(f"Mapped {len(map_data):,} of {len(shipwrecks_mapped):,} shipwrecks.")

In [ ]:
period_order = [
    "Before 1900",
    "1900–1938",
    "WWII (1939–1945)",
    "1946–1999",
    "2000 or later",
    "Unknown date",
]

period_colors = {
    "Before 1900": "#6a3d9a",
    "1900–1938": "#1f78b4",
    "WWII (1939–1945)": "#e31a1c",
    "1946–1999": "#ff7f00",
    "2000 or later": "#33a02c",
    "Unknown date": "#777777",
}

fig = px.scatter_geo(
    map_data,
    lat="Latitude",
    lon="Longitude",
    color="Date period",
    category_orders={"Date period": period_order},
    color_discrete_map=period_colors,
    hover_name="Ship",
    hover_data={
        "Flag": True,
        "Sunk date": True,
        "Coordinates": True,
        "Sinking year": True,
        "Latitude": False,
        "Longitude": False,
    },
    projection="natural earth",
    title="Mapped Shipwrecks by Sinking Period",
)
fig.update_traces(marker={"size": 6, "opacity": 0.72})
fig.update_geos(showland=True, landcolor="#efefef", showcountries=True)
fig.update_layout(height=650, legend_title_text="Sinking period")
fig.show()